# L12 — Project 2: Pose Recognition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yanluo/stem-on-stage-notebooks/blob/main/L12/02_pose_recognition/pose_starter.ipynb)

**Goal:** classify body posture (upright / bending / lying / tilted) from the micro:bit's pitch and roll readings.

**Why this project is different from L11.** L11 needed *time* — you had to look at a window of samples to tell walking from jumping. Pose is *static*. One reading is enough: pitch + roll fully describe orientation. No windowing, no machine learning needed for v1 — just `if/elif` rules.

**Where to run:** Google Colab. Each cell is `Shift + Enter`.

> **New to pandas / numpy?** Skim [`L12/00_python_data_tools/python_data_tools_starter.ipynb`](../00_python_data_tools/python_data_tools_starter.ipynb) first — it's a 30-minute tour of every function this notebook uses, with tiny standalone examples.

## Step 0 — Imports and a CSV loader

The pose CSV has just three columns: `t,pitch,roll` (degrees). The wearable program (`wearable_pose.py`) reads `input.rotation(Rotation.PITCH/ROLL)` directly — no math required.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("running in Colab" if IN_COLAB else "running locally")

## Step 1 — Load the sample recording

30 seconds, walking through five segments: upright → bend → upright → lying → upright → tilted → upright.

In [ ]:
SAMPLE_URL = "https://raw.githubusercontent.com/yanluo/stem-on-stage-notebooks/main/data/sample-pose.csv"

if IN_COLAB:
    df = pd.read_csv(SAMPLE_URL)
else:
    df = pd.read_csv("../../data/sample-pose.csv")

print(df.shape)
df.head()

## Step 2 — Plot pitch and roll over time

You should see four flat-ish regions separated by sharp transitions: the dancer holds a pose, then changes to the next.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(df["t"], df["pitch"], label="pitch", color="steelblue")
ax.plot(df["t"], df["roll"],  label="roll",  color="crimson")
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_xlabel("time (s)")
ax.set_ylabel("degrees")
ax.legend()
plt.show()

## Step 2.5 — Where do pitch and roll come from?

The micro:bit has no dedicated tilt sensor. It computes pitch and roll **from the accelerometer's reading of gravity**.

Here's the trick: when the board is at rest, the only force on the accelerometer is gravity (~1000 mg, straight down). If the board is flat with the logo up, that gravity reads on the board's `z` axis. **Tilt the board, and gravity now has components along multiple board axes** — and the relative size of those components tells you the tilt angle.

The processor uses `atan2(...)` (an inverse-tangent that handles all four quadrants correctly) to turn the (x, y, z) gravity reading into:

- **`Rotation.PITCH`** — forward/back tilt of the board, in degrees
- **`Rotation.ROLL`** — side-to-side tilt of the board, in degrees

You don't write the math — `input.rotation(Rotation.PITCH)` and `input.rotation(Rotation.ROLL)` give you the angles directly.

**Important caveat — the failure mode.** During fast motion (jumping, spinning), the accelerometer reads gravity **plus** acceleration. The "gravity vector" is contaminated, and pitch/roll briefly lie. **They're only reliable when the board is roughly still.** That's actually fine for *pose* recognition: holding a posture means the dancer is at rest, which is exactly when the angles are most accurate. It's why pose is a better fit for pitch/roll than, say, mid-jump orientation — for that you'd want a real gyroscope.

## Step 3 — A rule-based pose classifier

Pitch is the angle the board tips forward/back. Roll is the angle it tips side-to-side. The four poses fall in different regions of the (pitch, roll) plane:

| Pose            | Pitch              | Roll      |
|-----------------|--------------------|-----------|
| upright         | near 0             | near 0    |
| bending forward | large negative     | near 0    |
| lying on side   | near 0             | large     |
| tilted          | other              | other     |

The signs come from how the board is mounted: with the LED matrix facing up and the logo pointing forward (away from the body), tipping forward rotates the board so pitch goes *negative*. Twisting sideways drives roll positive or negative depending on which way you twist.

Translate the table into a function. **This is the same shape as the wearable program** — every line of `pose(pitch, roll)` below has a one-to-one Python equivalent on the device.

In [ ]:
def pose(pitch, roll):
    if abs(pitch) < 15 and abs(roll) < 15:
        return "upright"
    if pitch < -25:
        return "bending forward"
    if abs(roll) > 40:
        return "lying on side"
    return "tilted"

df["pose"] = [pose(p, r) for p, r in zip(df["pitch"], df["roll"])]
df["pose"].value_counts()

## Step 4 — See the classifier on top of the data

Color each sample by predicted pose. The colored bands should line up with the flat regions of the pitch/roll plot.

In [ ]:
POSE_COLORS = {
    "upright":          "#4caf50",
    "bending forward":  "#ff9800",
    "lying on side":    "#9c27b0",
    "tilted":           "#03a9f4",
}

fig, ax = plt.subplots(figsize=(10, 3.5))
for pose_name, color in POSE_COLORS.items():
    sub = df[df["pose"] == pose_name]
    ax.scatter(sub["t"], sub["pitch"], s=18, color=color, label=pose_name)
ax.set_xlabel("time (s)")
ax.set_ylabel("pitch (deg)")
ax.legend(loc="upper right", fontsize=8)
plt.show()

## Step 5 — A 2D view: pitch vs. roll

Plotting pitch on the x-axis and roll on the y-axis turns each sample into a point. You can *see* the regions the rules carve out — and pick better thresholds by eye if a class is in the wrong box.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for pose_name, color in POSE_COLORS.items():
    sub = df[df["pose"] == pose_name]
    ax.scatter(sub["pitch"], sub["roll"], s=18, color=color, label=pose_name)
# threshold lines from the classifier above
ax.axvline( 15, color="gray", linewidth=0.5, linestyle="--")
ax.axvline(-15, color="gray", linewidth=0.5, linestyle="--")
ax.axvline(-25, color="gray", linewidth=0.5, linestyle="--")
ax.axhline( 40, color="gray", linewidth=0.5, linestyle="--")
ax.axhline(-40, color="gray", linewidth=0.5, linestyle="--")
ax.set_xlabel("pitch (deg)")
ax.set_ylabel("roll (deg)")
ax.set_title("pose classification regions")
ax.legend(loc="upper right", fontsize=8)
plt.show()

## What to build next (L13 starting goals)

1. **Capture your own data.** Flash `wearable_pose.py` to a micro:bit, clip it to a clothing band, walk through the four poses, and record with the L11 datalogger pattern (extending it to log pitch/roll instead of x/y/z).
2. **Add an "arms-up" pose** by mounting the micro:bit on a wrist instead of the chest. The thresholds will need to change.
3. **Drive a NeoPixel color** from pose: upright = white, bending = blue, lying = off, tilted = yellow.
4. **Smooth the prediction** with a 1-second majority vote so brief stumbles don't flicker the lantern color.
5. **Per-dancer calibration:** record 5 s of each dancer's natural "upright" and store the offset. The thresholds become *relative* to that dancer's baseline.

## Reflect (homework)

Write 3–4 sentences in this cell:
- Which extension from the list will you build first, and why?
- What input/data do you need that you don't have yet?
- What's one thing you predict will be hard?